In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load the PDFs (metadata keeps the source path and page number)
docs = PyPDFDirectoryLoader("../documents", recursive=True).load()

# tag each chunk with its area
for d in docs:
    d.metadata["area"] = d.metadata["source"].split("/")[-2]

# 2. Chunk
chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_documents(docs)

# 3. Embed and store, persisted on disk
vs = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(api_key=OPENAI_API_KEY),
    persist_directory="../chroma_db",
)

# 4. Reload later without re-embedding
# vs = Chroma(persist_directory="../chroma_db", embedding_function=OpenAIEmbeddings())
retriever = vs.as_retriever(search_kwargs={"k": 8})

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.memory import ConversationBufferWindowMemory

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

memory = ConversationBufferWindowMemory(k=5, memory_key="history", return_messages=False)

def get_history(_):
    return memory.load_memory_variables({})["history"]


prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the IronStore enterprise assistant. Answer ONLY using the context below. "
     "If the answer is not in the context, say you don't know. "
     "Cite the source file name for each fact.\n\n"
     "Use the chat history only to understand what the user is referring to in follow-up "
     "questions (for example 'it', 'that', 'and for part-time staff?'). "
     "Facts in your answer must still come from the context, not from the history.\n\n"
     "Chat history:\n{history}\n\nContext:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    parts = []
    for d in docs:
        name = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page")
        label = f"{name}, p.{page + 1}" if isinstance(page, int) else name
        parts.append(f"[{label}]\n{d.page_content}")
    return "\n\n".join(parts)

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough(),
     "history": get_history}
    | prompt
    | llm
    | StrOutputParser()

)

def ask(question):
    answer = rag_chain.invoke(question)
    memory.save_context({"input": question}, {"output": answer})
    return answer

print(ask("How far in advance must I request annual leave longer than five working days?"))
print(ask("And how do I submit it?"))




You must submit annual leave requests for leave exceeding five working days at least 20 calendar days in advance. (IronStore_Leave_and_Absence_Policy.pdf, p.2)
Claims must be submitted in SpendHub within 30 calendar days of the expense date. Month-end claims should be submitted by the third working day of the following month. (IronStore_Expense_Reimbursement_Policy.pdf, p.5)


In [5]:
def run(questions):
    for q in questions:
        print(f"Q: {q}\nA: {ask(q)}\n{'-'*60}")

# 1. Single questions with known answers
run([
    "How long do I have to return a product?",                     # 30 calendar days
    "How long does a card refund usually take?",                             # 3-7 business days
    "What is the price of the Nordview NQ-55U7 TV?",                         # €649.00
    "Within how many days must expense claims be submitted, and in which system?",  # 30 calendar days, SpendHub
    "How quickly must a manager respond to a remote work request?",          # 15 calendar days
])

Q: How long do I have to return a product?
A: You have 14 calendar days to return a product after receiving a return authorization number (RMA) for approved returns. (IronStore_Returns_and_Reverse_Logistics_Procedure.pdf, p.2)
------------------------------------------------------------
Q: How long does a card refund usually take?
A: A card refund usually takes 3–7 business days after approval. (IronStore_Refund_and_Complaint_Handling_Policy.pdf, p.4)
------------------------------------------------------------
Q: What is the price of the Nordview NQ-55U7 TV?
A: The price of the Nordview NQ-55U7 55-inch 4K QLED Smart TV is €649.00. (IronStore_Product_Catalog_Electronics.pdf, p.1)
------------------------------------------------------------
Q: Within how many days must expense claims be submitted, and in which system?
A: Expense claims must be submitted within 30 calendar days of the expense date in SpendHub. (IronStore_Expense_Reimbursement_Policy.pdf, p.5)
----------------------------

In [6]:
# 2. Follow-ups: these test the memory
memory.clear()
run([
    "How long does a card refund usually take?",
    "And what about bank transfers?",              # 3-5 business days after bank details validated
    "Which of the two is faster?",                 # answered from history, bank transfer or card
])

Q: How long does a card refund usually take?
A: A card and digital wallet refund usually takes 3–7 business days after approval. [IronStore_Refund_and_Complaint_Handling_Policy.pdf, p.4]
------------------------------------------------------------
Q: And what about bank transfers?
A: I don't know.
------------------------------------------------------------
Q: Which of the two is faster?
A: I don't know.
------------------------------------------------------------


In [8]:
memory.clear()
run([
    "What equipment does IronStore provide for remote work?",
    "Who approves a remote work request?",
    "what did i ask?",           # up to three months
])

Q: What equipment does IronStore provide for remote work?
A: IronStore provides standard equipment based on role requirements for remote work. The normal package includes:

- Company laptop and charger.
- Headset and security key where required.
- External monitor for approved roles.
- Ergonomic accessories where supported by the local office or workplace assessment process. 

(Source: IronStore_Remote_Work_Policy.pdf, p.3)
------------------------------------------------------------
Q: Who approves a remote work request?
A: A remote work request is approved by the manager, who assesses role suitability, performance, team coverage, and customer impact. It also requires review by the relevant People Partner and a country-specific review where required by local employment law. (Source: IronStore_Remote_Work_Policy.pdf, p.2)
------------------------------------------------------------
Q: what did i ask?
A: I don't know.
------------------------------------------------------------


In [9]:
memory.clear()
run([
    "How long do I have to return aproduct?",
    "Does that apply to personalised goods too?",     # "that" = the 30-day return window
])

Q: How long do I have to return aproduct?
A: For approved returns, the return authorization number (RMA) is valid for 14 calendar days unless a different period is shown in the case. [IronStore_Returns_and_Reverse_Logistics_Procedure.pdf, p.2]
------------------------------------------------------------
Q: Does that apply to personalised goods too?
A: Personalised goods are normally non-returnable unless faulty. [IronStore_Refund_and_Complaint_Handling_Policy.pdf, p.3]
------------------------------------------------------------
